In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv = pd.read_csv(BASE_PATH + 'inventory_master.csv')
po_h = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
products = pd.read_csv(BASE_PATH + 'products.csv')

for name, df in [('ledger', ledger), ('inv', inv), ('po_h', po_h), ('po_l', po_l), ('so_h', so_h), ('so_l', so_l), ('products', products)]:
    print(name, "| rows:", len(df), "| columns:", len(df.columns))

Mounted at /content/drive
ledger | rows: 237230 | columns: 9
inv | rows: 180 | columns: 8
po_h | rows: 24000 | columns: 10
po_l | rows: 155495 | columns: 9
so_h | rows: 20000 | columns: 11
so_l | rows: 130402 | columns: 9
products | rows: 30 | columns: 23


In [2]:
print("Products rows:", len(products), "| unit_cost empty:", products['unit_cost'].isnull().sum(), "| zero or negative:", (products['unit_cost'] <= 0).sum())
print("Products with unit_cost at or above unit_price:", (products['unit_cost'] >= products['unit_price']).sum())

chk = products[['product_id', 'product_name', 'uom', 'unit_cost', 'unit_price']].copy()
chk['cost_share_of_price_%'] = (chk['unit_cost'] / chk['unit_price'] * 100).round(1)
print(chk.sort_values('unit_cost').to_string(index=False))

m = po_l.merge(products[['product_id', 'unit_cost']], on='product_id', how='left', suffixes=('_po', '_product'))
print("\nPO lines:", len(po_l), "| rows after merge:", len(m))
print("PO lines with no matching product:", m['unit_cost_product'].isnull().sum())
print("PO lines where unit_cost differs from the product's unit_cost:", (m['unit_cost_po'] != m['unit_cost_product']).sum())

Products rows: 30 | unit_cost empty: 0 | zero or negative: 0
Products with unit_cost at or above unit_price: 0
product_id                 product_name       uom  unit_cost  unit_price  cost_share_of_price_%
      P026        Grease Cartridge GC-5 cartridge        180         310                   58.1
      P027             O-Ring Set OR-50       set        260         480                   54.2
      P017               Fan Belt FB-30     piece        320         520                   61.5
      P008           Bucket Tooth BT-10     piece        340         580                   58.6
      P002      Engine Oil Filter OF-90     piece        450         720                   62.5
      P020              Pivot Pin PP-12     piece        750        1150                   65.2
      P003            Air Filter AF-120     piece        820        1250                   65.6
      P013               Seal Kit SK-15       set        950        1450                   65.5
      P007         Hydrau

In [3]:
m['diff_%'] = (m['unit_cost_po'] / m['unit_cost_product'] - 1) * 100
print("PO lines:", len(m))
print("PO cost higher than product cost:", (m['diff_%'] > 0).sum(), "| lower:", (m['diff_%'] < 0).sum(), "| equal:", (m['diff_%'] == 0).sum())
print("\nDifference in % per line (PO cost compared with product cost):")
print(m['diff_%'].describe().round(2).to_string())

per = m.groupby('product_id').agg(product_cost=('unit_cost_product', 'first'), po_min=('unit_cost_po', 'min'), po_mean=('unit_cost_po', 'mean'), po_max=('unit_cost_po', 'max'), lines=('unit_cost_po', 'size')).reset_index()
per['po_mean'] = per['po_mean'].round(1)
per['mean_vs_product_%'] = ((per['po_mean'] / per['product_cost'] - 1) * 100).round(2)
print("\nPer product:")
print(per.to_string(index=False))

PO lines: 155495
PO cost higher than product cost: 94007 | lower: 61485 | equal: 3

Difference in % per line (PO cost compared with product cost):
count    155495.00
mean          6.11
std          17.31
min         -27.54
25%          -6.83
50%           4.64
75%          16.39
max          66.15

Per product:
product_id  product_cost   po_min  po_mean    po_max  lines  mean_vs_product_%
      P001         18950 14701.95  18376.3  22048.91   5183              -3.03
      P002           450   432.16    540.2    647.99   5266              20.04
      P003           820   750.02    938.9   1124.78   5234              14.50
      P004         32500 23881.60  29811.7  35818.17   5132              -8.27
      P005          6350  5070.13   6336.0   7604.51   5276              -0.22
      P006          8650  6720.28   8404.3  10078.10   5170              -2.84
      P007           980   870.01   1090.8   1305.00   5238              11.31
      P008           340   348.05    435.9    521.99   

In [4]:
mm = m.merge(po_h[['po_id', 'order_date']], on='po_id', how='left')
print("Rows after merge:", len(mm), "| lines with no header date:", mm['order_date'].isnull().sum())
mm['year'] = pd.to_datetime(mm['order_date']).dt.year
print("\nAverage difference in % (PO cost vs product cost) by order year:")
print(mm.groupby('year')['diff_%'].agg(['count', 'mean']).round(2).to_string())

r = per['po_max'] / per['po_min']
print("\nHighest PO cost / lowest PO cost per product: smallest", round(r.min(), 3), "| largest", round(r.max(), 3))

Rows after merge: 155495 | lines with no header date: 0

Average difference in % (PO cost vs product cost) by order year:
      count  mean
year             
2019  25802  6.24
2020  26568  6.16
2021  25951  6.17
2022  25515  6.02
2023  26020  6.05
2024  25639  6.00

Highest PO cost / lowest PO cost per product: smallest 1.499 | largest 1.5


In [5]:
rec_ids = po_h.loc[po_h['po_status'] == 'Received', 'po_id']
rl = po_l[po_l['po_id'].isin(rec_ids)]
print("Table used: PO lines of Received orders | lines:", len(rl), "| units:", rl['quantity'].sum())
print("Lines where line_total differs from quantity x unit_cost by more than 0.01:", ((rl['quantity'] * rl['unit_cost'] - rl['line_total']).abs() > 0.01).sum())

w = rl.groupby('product_id').agg(units=('quantity', 'sum'), spend=('line_total', 'sum')).reset_index()
w['avg_cost_received'] = (w['spend'] / w['units']).round(2)
w = w.merge(products[['product_id', 'unit_cost']].rename(columns={'unit_cost': 'products_table_cost'}), on='product_id')
w = w.merge(per[['product_id', 'po_mean']].rename(columns={'po_mean': 'simple_mean_all_lines'}), on='product_id')
w['vs_products_table_%'] = ((w['avg_cost_received'] / w['products_table_cost'] - 1) * 100).round(2)
print("Products:", len(w), "| units total:", w['units'].sum(), "| spend total:", w['spend'].sum())
print(w[['product_id', 'units', 'avg_cost_received', 'simple_mean_all_lines', 'products_table_cost', 'vs_products_table_%']].to_string(index=False))

Table used: PO lines of Received orders | lines: 140077 | units: 22387123
Lines where line_total differs from quantity x unit_cost by more than 0.01: 0
Products: 30 | units total: 22387123 | spend total: 312027429107.6999
product_id  units  avg_cost_received  simple_mean_all_lines  products_table_cost  vs_products_table_%
      P001 749292           18380.34                18376.3                18950                -3.01
      P002 764701             540.40                  540.2                  450                20.09
      P003 754421             938.35                  938.9                  820                14.43
      P004 740161           29785.03                29811.7                32500                -8.35
      P005 759066            6332.77                 6336.0                 6350                -0.27
      P006 751495            8405.06                 8404.3                 8650                -2.83
      P007 755069            1091.81                 1090.8     

In [6]:
sd = so_l.merge(so_h[['so_id', 'branch_id', 'delivery_date', 'order_status']], on='so_id', how='left')
sd = sd[sd['order_status'] == 'Delivered'].copy()
sd['year'] = pd.to_datetime(sd['delivery_date']).dt.year

pr = po_l.merge(po_h[['po_id', 'branch_id', 'received_date', 'po_status']], on='po_id', how='left')
pr = pr[pr['po_status'] == 'Received'].copy()
pr['year'] = pd.to_datetime(pr['received_date']).dt.year

print("Delivered lines:", len(sd), "| units:", sd['quantity'].sum(), "| lines with no delivery date:", sd['delivery_date'].isnull().sum())
print("Received lines:", len(pr), "| units:", pr['quantity'].sum(), "| lines with no received date:", pr['received_date'].isnull().sum())

y = pd.DataFrame({'delivered_units': sd.groupby('year')['quantity'].sum(), 'received_units': pr.groupby('year')['quantity'].sum()}).fillna(0).astype(int)
print("\nUnits by calendar year:")
print(y.to_string())
print("\nTotals of the yearly rows:", y['delivered_units'].sum(), "|", y['received_units'].sum())

Delivered lines: 117656 | units: 1234939 | lines with no delivery date: 0
Received lines: 140077 | units: 22387123 | lines with no received date: 0

Units by calendar year:
      delivered_units  received_units
year                                 
2019           204586         3556945
2020           211190         3786822
2021           206935         3776259
2022           202716         3682130
2023           205578         3752938
2024           200562         3637075
2025             3372          194954

Totals of the yearly rows: 1234939 | 22387123


In [7]:
L = ledger.merge(inv[['product_id', 'branch_id', 'opening_stock']], on=['product_id', 'branch_id'], how='left')
L['prev'] = L.groupby(['product_id', 'branch_id'])['running_balance'].shift(1).fillna(L['opening_stock'])
L['change'] = L['running_balance'] - L['prev']
print("Ledger rows:", len(L))
print("IN rows where change = +quantity:", ((L['movement_type'] == 'IN') & (L['change'] == L['quantity'])).sum(), "of", (L['movement_type'] == 'IN').sum())
print("OUT rows where change = -quantity:", ((L['movement_type'] == 'OUT') & (L['change'] == -L['quantity'])).sum(), "of", (L['movement_type'] == 'OUT').sum())
print("ADJUSTMENT rows where change = +/- quantity:", ((L['movement_type'] == 'ADJUSTMENT') & (L['change'].abs() == L['quantity'])).sum(), "of", (L['movement_type'] == 'ADJUSTMENT').sum())

adj = L[L['movement_type'] == 'ADJUSTMENT']
f1 = pr[['product_id', 'branch_id', 'received_date', 'quantity']].rename(columns={'received_date': 'date'})
f2 = sd[['product_id', 'branch_id', 'delivery_date', 'quantity']].rename(columns={'delivery_date': 'date'})
f2['quantity'] = -f2['quantity']
f3 = adj[['product_id', 'branch_id', 'movement_date', 'change']].rename(columns={'movement_date': 'date', 'change': 'quantity'})
fl = pd.concat([f1, f2, f3], ignore_index=True)
fl['month'] = pd.to_datetime(fl['date']).dt.to_period('M')

pairs = inv.set_index(['product_id', 'branch_id'])
months = pd.period_range('2019-01', '2025-01', freq='M')
print("\nMovements used:", len(fl), "| outside the 73 months:", (~fl['month'].isin(months)).sum())
net = fl.groupby(['product_id', 'branch_id', 'month'])['quantity'].sum().unstack('month')
net = net.reindex(index=pairs.index, columns=months).fillna(0)
stock = net.cumsum(axis=1).add(pairs['opening_stock'], axis=0)

print("Pairs:", len(stock), "| months:", stock.shape[1])
print("Total stock at end of", months[-1], ":", int(stock[months[-1]].sum()), "(KPI 8 order-line stock: 21185215)")
print("Pair-months below zero:", int((stock < 0).sum().sum()), "| smallest value:", int(stock.min().min()))
for y in range(2019, 2025):
    print("Total stock at end of Dec", y, ":", int(stock[pd.Period(str(y) + '-12', freq='M')].sum()))

Ledger rows: 237230
IN rows where change = +quantity: 127611 of 127611
OUT rows where change = -quantity: 107165 of 107165
ADJUSTMENT rows where change = +/- quantity: 2454 of 2454

Movements used: 260187 | outside the 73 months: 0
Pairs: 180 | months: 73
Total stock at end of 2025-01 : 21185215 (KPI 8 order-line stock: 21185215)
Pair-months below zero: 0 | smallest value: 67
Total stock at end of Dec 2019 : 3384660
Total stock at end of Dec 2020 : 6960247
Total stock at end of Dec 2021 : 10529943
Total stock at end of Dec 2022 : 14009405
Total stock at end of Dec 2023 : 17557387
Total stock at end of Dec 2024 : 20993633


In [8]:
yrs = list(range(2019, 2025))
sold = sd.groupby(['product_id', 'branch_id', 'year'])['quantity'].sum().unstack('year').reindex(index=pairs.index, columns=yrs).fillna(0)

avgA = pd.DataFrame({y: stock[[pd.Period(f'{y}-{m:02d}', freq='M') for m in range(1, 13)]].mean(axis=1) for y in yrs})
start = {y: (pairs['opening_stock'] if y == 2019 else stock[pd.Period(f'{y-1}-12', freq='M')]) for y in yrs}
avgB = pd.DataFrame({y: (start[y] + stock[pd.Period(f'{y}-12', freq='M')]) / 2 for y in yrs})

cw = w.set_index('product_id')['avg_cost_received']
ct = products.set_index('product_id')['unit_cost']
prod = pairs.index.get_level_values('product_id')

rows = []
for y in yrs:
    r = {'year': y, 'units_sold': int(sold[y].sum()), 'avg_stock_units': int(avgA[y].sum())}
    for cname, c in [('received_cost', cw), ('table_cost', ct)]:
        cv = c.reindex(prod).values
        for mname, avg in [('monthly_avg', avgA), ('open_close', avgB)]:
            r[cname + ' | ' + mname] = round((sold[y].values * cv).sum() / (avg[y].values * cv).sum(), 5)
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))
print("\nTotal units sold 2019 to 2024:", int(sold.sum().sum()))

 year  units_sold  avg_stock_units  received_cost | monthly_avg  received_cost | open_close  table_cost | monthly_avg  table_cost | open_close
 2019      204586          1798354                      0.11418                     0.12003                   0.11419                  0.12003
 2020      211190          5302595                      0.03963                     0.04053                   0.03962                  0.04052
 2021      206935          8899310                      0.02321                     0.02360                   0.02320                  0.02359
 2022      202716         12378411                      0.01666                     0.01683                   0.01666                  0.01684
 2023      205578         15890688                      0.01301                     0.01309                   0.01301                  0.01309
 2024      200562         19389482                      0.01023                     0.01029                   0.01022                  0.01028

In [9]:
cv = cw.reindex(prod).values
tab = {}
for y in yrs:
    num = pd.Series(sold[y].values * cv, index=pairs.index).groupby(level='branch_id').sum()
    den = pd.Series(avgA[y].values * cv, index=pairs.index).groupby(level='branch_id').sum()
    tab[y] = (num / den).round(5)
print("Turnover by branch and year (received cost, monthly average):")
print(pd.DataFrame(tab).to_string())

pt = sold[2024] / avgA[2024]
print("\nTurnover 2024 per product-branch pair (units sold / average stock), 180 pairs:")
print(pt.describe().round(5).to_string())
print("\nSlowest 5 pairs:")
print(pt.sort_values().head(5).round(5).to_string())
print("\nFastest 5 pairs:")
print(pt.sort_values().tail(5).round(5).to_string())

Turnover by branch and year (received cost, monthly average):
              2019     2020     2021     2022     2023     2024
branch_id                                                      
AHM001     0.12899  0.04328  0.02476  0.01662  0.01450  0.00980
CHN001     0.15269  0.04355  0.02833  0.02005  0.01577  0.01222
DEL001     0.08946  0.03489  0.02131  0.01682  0.01143  0.00938
HYD001     0.08670  0.03269  0.01682  0.01413  0.00994  0.00856
KOL001     0.13151  0.04908  0.02979  0.01878  0.01563  0.01232
PUN001     0.10017  0.03526  0.01897  0.01380  0.01100  0.00917

Turnover 2024 per product-branch pair (units sold / average stock), 180 pairs:
count    180.00000
mean       0.01038
std        0.00186
min        0.00634
25%        0.00899
50%        0.01026
75%        0.01161
max        0.01530

Slowest 5 pairs:
product_id  branch_id
P005        HYD001       0.00634
P004        HYD001       0.00699
P019        HYD001       0.00699
P015        PUN001       0.00729
P018        HYD001    

In [10]:
b = pd.DataFrame({
    'units_sold_2024': sold[2024].groupby(level='branch_id').sum(),
    'units_received_2024': pr[pr['year'] == 2024].groupby('branch_id')['quantity'].sum(),
    'avg_stock_2024': avgA[2024].groupby(level='branch_id').sum(),
    'stock_end_2024': stock[pd.Period('2024-12', freq='M')].groupby(level='branch_id').sum()
})
b['turnover_units'] = (b['units_sold_2024'] / b['avg_stock_2024']).round(5)
b['years_of_stock'] = (1 / b['turnover_units']).round(1)
print(b.round(0).to_string())
print("\nTotals:", int(b['units_sold_2024'].sum()), "|", int(b['units_received_2024'].sum()))

           units_sold_2024  units_received_2024  avg_stock_2024  stock_end_2024  turnover_units  years_of_stock
branch_id                                                                                                      
AHM001             31921.0               615796       3165998.0       3442316.0             0.0            99.0
CHN001             40818.0               642929       3234502.0       3516571.0             0.0            79.0
DEL001             31967.0               627088       3293798.0       3566310.0             0.0           103.0
HYD001             27816.0               576606       3268726.0       3517178.0             0.0           118.0
KOL001             36941.0               568941       3147293.0       3400815.0             0.0            85.0
PUN001             31099.0               605715       3279165.0       3550443.0             0.0           106.0

Totals: 200562 | 3637075


In [11]:
pp = pd.DataFrame({
    'sold_2019': sold[2019].groupby(level='product_id').sum(),
    'avg_stock_2019': avgA[2019].groupby(level='product_id').sum(),
    'sold_2024': sold[2024].groupby(level='product_id').sum(),
    'avg_stock_2024': avgA[2024].groupby(level='product_id').sum()
})
pp['turnover_2019'] = (pp['sold_2019'] / pp['avg_stock_2019']).round(5)
pp['turnover_2024'] = (pp['sold_2024'] / pp['avg_stock_2024']).round(5)
pp['years_of_stock_2024'] = (1 / pp['turnover_2024']).round(1)
pp['avg_stock_2024'] = pp['avg_stock_2024'].round(0).astype(int)
pp['sold_2024'] = pp['sold_2024'].astype(int)
pp['name'] = products.set_index('product_id')['product_name']
print(pp[['name', 'sold_2024', 'avg_stock_2024', 'turnover_2024', 'years_of_stock_2024', 'turnover_2019']].to_string())
print("\nTotals 2024: units sold", int(pp['sold_2024'].sum()), "| average stock", int(pp['avg_stock_2024'].sum()))
print("Products where 2024 turnover is below 2019 turnover:", int((pp['turnover_2024'] < pp['turnover_2019']).sum()), "of", len(pp))

                                    name  sold_2024  avg_stock_2024  turnover_2024  years_of_stock_2024  turnover_2019
product_id                                                                                                            
P001               Hydraulic Pump HP-300       6636          645793        0.01028                 97.3        0.11318
P002             Engine Oil Filter OF-90       7145          664365        0.01075                 93.0        0.11230
P003                   Air Filter AF-120       6728          649329        0.01036                 96.5        0.10626
P004                   Track Chain TC-45       6668          639256        0.01043                 95.9        0.12962
P005                Fuel Injector FI-220       6137          658617        0.00932                107.3        0.11501
P006                  Alternator ALT-500       7192          651481        0.01104                 90.6        0.09492
P007                Hydraulic Hose HH-60       6

In [12]:
print(b[['units_sold_2024', 'avg_stock_2024', 'turnover_units', 'years_of_stock']].round({'units_sold_2024': 0, 'avg_stock_2024': 0, 'turnover_units': 5, 'years_of_stock': 1}).to_string())

           units_sold_2024  avg_stock_2024  turnover_units  years_of_stock
branch_id                                                                 
AHM001             31921.0       3165998.0         0.01008            99.2
CHN001             40818.0       3234502.0         0.01262            79.2
DEL001             31967.0       3293798.0         0.00971           103.0
HYD001             27816.0       3268726.0         0.00851           117.5
KOL001             36941.0       3147293.0         0.01174            85.2
PUN001             31099.0       3279165.0         0.00948           105.5


In [13]:
print("Table used: sd (Delivered sales lines) | rows:", len(sd))
for c in ['so_id', 'product_id', 'branch_id', 'delivery_date', 'quantity']:
    print(" ", c, "| empty:", sd[c].isnull().sum())
print("  quantity | min", sd['quantity'].min(), "| zeros", (sd['quantity'] == 0).sum(), "| negatives", (sd['quantity'] < 0).sum())

print("\nTable used: pr (Received PO lines) | rows:", len(pr))
for c in ['po_id', 'product_id', 'branch_id', 'received_date', 'quantity', 'unit_cost', 'line_total']:
    print(" ", c, "| empty:", pr[c].isnull().sum())
for c in ['quantity', 'unit_cost', 'line_total']:
    print(" ", c, "| min", pr[c].min(), "| zeros", (pr[c] == 0).sum(), "| negatives", (pr[c] < 0).sum())

print("\nTable used: inv (opening stock) | rows:", len(inv))
print("  opening_stock | empty:", inv['opening_stock'].isnull().sum(), "| min", inv['opening_stock'].min(), "| zeros", (inv['opening_stock'] == 0).sum(), "| negatives", (inv['opening_stock'] < 0).sum())

print("\nTable used: ledger (corrections and balances) | rows:", len(ledger))
for c in ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'running_balance']:
    print(" ", c, "| empty:", ledger[c].isnull().sum())
print("  quantity | zeros", (ledger['quantity'] == 0).sum(), "| negatives", (ledger['quantity'] < 0).sum(), "| running_balance negatives", (ledger['running_balance'] < 0).sum())

print("\nTable used: products (cost) | rows:", len(products), "| unit_cost empty:", products['unit_cost'].isnull().sum())

Table used: sd (Delivered sales lines) | rows: 117656
  so_id | empty: 0
  product_id | empty: 0
  branch_id | empty: 0
  delivery_date | empty: 0
  quantity | empty: 0
  quantity | min 1 | zeros 0 | negatives 0

Table used: pr (Received PO lines) | rows: 140077
  po_id | empty: 0
  product_id | empty: 0
  branch_id | empty: 0
  received_date | empty: 0
  quantity | empty: 0
  unit_cost | empty: 0
  line_total | empty: 0
  quantity | min 20 | zeros 0 | negatives 0
  unit_cost | min 186.05 | zeros 0 | negatives 0
  line_total | min 3763.6 | zeros 0 | negatives 0

Table used: inv (opening stock) | rows: 180
  opening_stock | empty: 0 | min 80 | zeros 0 | negatives 0

Table used: ledger (corrections and balances) | rows: 237230
  product_id | empty: 0
  branch_id | empty: 0
  movement_type | empty: 0
  movement_date | empty: 0
  quantity | empty: 0
  running_balance | empty: 0
  quantity | zeros 0 | negatives 0 | running_balance negatives 0

Table used: products (cost) | rows: 30 | unit_c

In [14]:
tables = {'sd (Delivered sales lines)': sd, 'pr (Received PO lines)': pr,
          'inv': inv, 'ledger': ledger, 'products': products}

for name, t in tables.items():
    print('\n' + name, '| rows:', len(t))
    nums = t.select_dtypes('number')
    out = pd.DataFrame({'empty': nums.isna().sum(),
                        'zero': (nums == 0).sum(),
                        'negative': (nums < 0).sum()})
    print(out)


sd (Delivered sales lines) | rows: 117656
                  empty  zero  negative
line_number           0     0         0
quantity              0     0         0
unit_price            0     0         0
gst_rate              0     0         0
line_total            0     0         0
gst_amount            0     0         0
line_grand_total      0     0         0
year                  0     0         0

pr (Received PO lines) | rows: 140077
                  empty  zero  negative
line_number           0     0         0
quantity              0     0         0
unit_cost             0     0         0
gst_rate              0     0         0
line_total            0     0         0
gst_amount            0     0         0
line_grand_total      0     0         0
year                  0     0         0

inv | rows: 180
               empty  zero  negative
opening_stock      0     0         0
reorder_level      0     0         0
safety_stock       0     0         0
max_stock          0     0       

In [15]:
idx = pd.MultiIndex.from_frame(inv[['product_id', 'branch_id']])
print('pairs:', len(idx), '| pair-years expected:', len(idx) * 6)

for name, t in [('sd (Delivered sales lines)', sd), ('pr (Received PO lines)', pr)]:
    t6 = t[t['year'].between(2019, 2024)]
    units = (t6.groupby(['product_id', 'branch_id', 'year'])['quantity'].sum()
               .unstack('year').reindex(idx).reindex(columns=range(2019, 2025)).fillna(0))
    print('\n' + name, '| rows used (2019-2024):', len(t6))
    print('pair-years:', units.size, '| with zero units:', int((units == 0).sum().sum()),
          '| smallest units in a pair-year:', int(units.min().min()))

print('\nstock (month-end order-line stock) | smallest value of any pair-month:', int(stock.min().min()))

pairs: 180 | pair-years expected: 1080

sd (Delivered sales lines) | rows used (2019-2024): 117316
pair-years: 1080 | with zero units: 0 | smallest units in a pair-year: 593

pr (Received PO lines) | rows used (2019-2024): 138831
pair-years: 1080 | with zero units: 0 | smallest units in a pair-year: 13663

stock (month-end order-line stock) | smallest value of any pair-month: 67


In [16]:
for name, t, oid in [('sd (Delivered sales lines)', sd, 'so_id'), ('pr (Received PO lines)', pr, 'po_id')]:
    print('\n' + name, '| rows:', len(t))
    print('repeated (', oid, ', line_number ) keys:', int(t.duplicated([oid, 'line_number'], keep=False).sum()))
    print('rows that are exact copies of another row:', int(t.duplicated(keep=False).sum()))
    print('same product twice in one order (rows):', int(t.duplicated([oid, 'product_id'], keep=False).sum()))


sd (Delivered sales lines) | rows: 117656
repeated ( so_id , line_number ) keys: 0
rows that are exact copies of another row: 0
same product twice in one order (rows): 25422

pr (Received PO lines) | rows: 140077
repeated ( po_id , line_number ) keys: 0
rows that are exact copies of another row: 0
same product twice in one order (rows): 30068


In [17]:
for name, t, oid, pcol in [('sd (Delivered sales lines)', sd, 'so_id', 'unit_price'),
                           ('pr (Received PO lines)', pr, 'po_id', 'unit_cost')]:
    print('\n' + name, '| rows:', len(t))
    rep = t[t.duplicated([oid, 'product_id'], keep=False)]
    print('rows where the product is repeated in the order:', len(rep))
    same = rep.duplicated([oid, 'product_id', 'quantity', pcol], keep=False)
    print('of these, same quantity and same', pcol, '(only line_number differs):', int(same.sum()))
    q = rep['quantity']
    print('quantity range of these rows: min', int(q.min()), '| max', int(q.max()), '| different quantities:', q.nunique())
    print('chance of two lines having the same quantity if independent (about 1 / different quantities):',
          round(1 / q.nunique(), 4))


sd (Delivered sales lines) | rows: 117656
rows where the product is repeated in the order: 25422
of these, same quantity and same unit_price (only line_number differs): 1374
quantity range of these rows: min 1 | max 20 | different quantities: 20
chance of two lines having the same quantity if independent (about 1 / different quantities): 0.05

pr (Received PO lines) | rows: 140077
rows where the product is repeated in the order: 30068
of these, same quantity and same unit_cost (only line_number differs): 0
quantity range of these rows: min 20 | max 300 | different quantities: 281
chance of two lines having the same quantity if independent (about 1 / different quantities): 0.0036


In [18]:
rep = sd[sd.duplicated(['so_id', 'product_id'], keep=False)]
g = rep.groupby(['so_id', 'product_id']).size()
print('sd | rows in repeated groups:', len(rep), '| groups (order + product):', len(g))
print('group sizes (lines per group):')
print(g.value_counts().sort_index())

obs = rep.groupby(['so_id', 'product_id', 'quantity']).size()
obs_pairs = int((obs * (obs - 1) // 2).sum())
all_pairs = int((g * (g - 1) // 2).sum())
p = rep['quantity'].value_counts(normalize=True)
same_prob = float((p ** 2).sum())

print('pairs of lines inside the groups:', all_pairs)
print('chance that a pair has the same quantity (from the real spread):', round(same_prob, 4))
print('pairs with the same quantity, expected by chance:', round(all_pairs * same_prob, 1))
print('pairs with the same quantity, observed:', obs_pairs)

sd | rows in repeated groups: 25422 | groups (order + product): 12184
group sizes (lines per group):
2    11202
3      914
4       65
5        2
6        1
Name: count, dtype: int64
pairs of lines inside the groups: 14369
chance that a pair has the same quantity (from the real spread): 0.0501
pairs with the same quantity, expected by chance: 719.3
pairs with the same quantity, observed: 690


In [19]:
print('sd rows:', len(sd), '| pr rows:', len(pr))
print('units delivered:', int(sd['quantity'].sum()), '(expected 1234939)')
print('units received :', int(pr['quantity'].sum()), '(expected 22387123)')
print('sd lines where line_total differs from quantity x unit_price (more than 0.01):',
      int(((sd['line_total'] - sd['quantity'] * sd['unit_price']).abs() > 0.01).sum()))
print('pr lines where line_total differs from quantity x unit_cost (more than 0.01):',
      int(((pr['line_total'] - pr['quantity'] * pr['unit_cost']).abs() > 0.01).sum()))

adj_net = int(L.loc[L['reference_id'].astype(str).str.startswith('ADJ'), 'change'].sum())
rebuilt = int(inv['opening_stock'].sum() + pr['quantity'].sum() - sd['quantity'].sum() + adj_net)
last = stock.iloc[:, -1].sum() if stock.shape[0] == 180 else stock.iloc[-1].sum()
print('\nopening stock (inv):', int(inv['opening_stock'].sum()), '| adjustments, net:', adj_net)
print('opening + received - delivered + adjustments =', rebuilt)
print('stock table, last month (2025-01), all pairs =', int(last))
print('KPI 8 order-line stock (expected)            = 21185215')

sd rows: 117656 | pr rows: 140077
units delivered: 1234939 (expected 1234939)
units received : 22387123 (expected 22387123)
sd lines where line_total differs from quantity x unit_price (more than 0.01): 0
pr lines where line_total differs from quantity x unit_cost (more than 0.01): 0

opening stock (inv): 32969 | adjustments, net: 62
opening + received - delivered + adjustments = 21185215
stock table, last month (2025-01), all pairs = 21185215
KPI 8 order-line stock (expected)            = 21185215


In [20]:
s = sd.assign(m=pd.to_datetime(sd['delivery_date']).dt.to_period('M'))
r = pr.assign(m=pd.to_datetime(pr['received_date']).dt.to_period('M'))
mon = pd.period_range('2019-01', '2024-12', freq='M')

flow = (r.groupby('m')['quantity'].sum().reindex(mon).fillna(0)
        - s.groupby('m')['quantity'].sum().reindex(mon).fillna(0))
end_stock = int(inv['opening_stock'].sum()) + flow.cumsum()
avg2 = end_stock.groupby(end_stock.index.year).mean()
sold2 = s[s['m'].isin(mon)].groupby(s['m'].dt.year)['quantity'].sum()

step5 = {2019: 1798354, 2020: 5302595, 2021: 8899310, 2022: 12378411, 2023: 15890688, 2024: 19389482}
print('second method: total month-end stock from sd + pr + opening stock (no adjustments, no stock table)')
print('year | units sold | average stock, second method | average stock, Step 5 | gap % | turnover, second method | turnover, Step 5')
for y in range(2019, 2025):
    a2, a5 = avg2[y], step5[y]
    print(y, '|', int(sold2[y]), '|', int(round(a2)), '|', a5, '|', round((a2 - a5) / a5 * 100, 3), '|',
          round(sold2[y] / a2, 5), '|', round(sold2[y] / a5, 5))

second method: total month-end stock from sd + pr + opening stock (no adjustments, no stock table)
year | units sold | average stock, second method | average stock, Step 5 | gap % | turnover, second method | turnover, Step 5
2019 | 204586 | 1798915 | 1798354 | 0.031 | 0.11373 | 0.11376
2020 | 211190 | 5303361 | 5302595 | 0.014 | 0.03982 | 0.03983
2021 | 206935 | 8899554 | 8899310 | 0.003 | 0.02325 | 0.02325
2022 | 202716 | 12378860 | 12378411 | 0.004 | 0.01638 | 0.01638
2023 | 205578 | 15890459 | 15890688 | -0.001 | 0.01294 | 0.01294
2024 | 200562 | 19389511 | 19389482 | 0.0 | 0.01034 | 0.01034


In [21]:
st = stock if stock.shape[0] == 180 else stock.T
yr = pd.PeriodIndex(st.columns, freq='M').year
avg = st.T.groupby(yr).mean().T
sold_py = (sd[sd['year'].between(2019, 2024)]
           .groupby(['product_id', 'branch_id', 'year'])['quantity'].sum().unstack('year'))
sold_py = sold_py.reindex(st.index)
print('pairs:', len(st), '| pairs with no sales row after matching:', int(sold_py.isna().all(axis=1).sum()))

yrs6 = list(range(2019, 2025))
turn = sold_py[yrs6] / avg[yrs6]

print('\nyear | pairs | min | median | max | lower limit | upper limit | pairs below | pairs above')
for y in yrs6:
    v = turn[y]
    q1, q3 = v.quantile(0.25), v.quantile(0.75)
    lo, hi = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    print(y, '|', v.notna().sum(), '|', round(v.min(), 5), '|', round(v.median(), 5), '|', round(v.max(), 5),
          '|', round(lo, 5), '|', round(hi, 5), '|', int((v < lo).sum()), '|', int((v > hi).sum()))

print('\n2024 tie to Step 6 (expected mean 0.01038, min 0.00634, max 0.01530):')
print('mean', round(turn[2024].mean(), 5), '| min', round(turn[2024].min(), 5), '| max', round(turn[2024].max(), 5))

pairs: 180 | pairs with no sales row after matching: 0

year | pairs | min | median | max | lower limit | upper limit | pairs below | pairs above
2019 | 180 | 0.05804 | 0.11159 | 0.20234 | 0.02952 | 0.20049 | 0 | 1
2020 | 180 | 0.02233 | 0.03926 | 0.06249 | 0.01729 | 0.06198 | 0 | 1
2021 | 180 | 0.01361 | 0.02305 | 0.04156 | 0.00764 | 0.03877 | 0 | 1
2022 | 180 | 0.0096 | 0.01637 | 0.02699 | 0.00738 | 0.02533 | 0 | 1
2023 | 180 | 0.00666 | 0.0126 | 0.02076 | 0.0038 | 0.02221 | 0 | 0
2024 | 180 | 0.00634 | 0.01026 | 0.0153 | 0.00505 | 0.01555 | 0 | 0

2024 tie to Step 6 (expected mean 0.01038, min 0.00634, max 0.01530):
mean 0.01038 | min 0.00634 | max 0.0153


In [22]:
for y in yrs6:
    v = turn[y]
    q1, q3 = v.quantile(0.25), v.quantile(0.75)
    hi = q3 + 1.5 * (q3 - q1)
    out = v[v > hi]
    for pair_id, val in out.items():
        print(y, pair_id, '| turnover', round(val, 5), '| upper limit', round(hi, 5),
              '| units sold', int(sold_py.loc[pair_id, y]), '| average stock', int(round(avg.loc[pair_id, y])),
              '| typical pair: units sold', int(sold_py[y].median()), ', average stock', int(round(avg[y].median())))
    keep = v <= hi
    print(y, ' all pairs:', round(sold_py[y].sum() / avg[y].sum(), 5),
          '| without pairs above the limit:', round(sold_py.loc[keep, y].sum() / avg.loc[keep, y].sum(), 5))

2019 ('P001', 'CHN001') | turnover 0.20234 | upper limit 0.20049 | units sold 1689 | average stock 8347 | typical pair: units sold 1083 , average stock 9990
2019  all pairs: 0.11376 | without pairs above the limit: 0.11335
2020 ('P003', 'AHM001') | turnover 0.06249 | upper limit 0.06198 | units sold 1576 | average stock 25220 | typical pair: units sold 1175 , average stock 29415
2020  all pairs: 0.03983 | without pairs above the limit: 0.03972
2021 ('P021', 'KOL001') | turnover 0.04156 | upper limit 0.03877 | units sold 1771 | average stock 42609 | typical pair: units sold 1125 , average stock 49563
2021  all pairs: 0.02325 | without pairs above the limit: 0.02316
2022 ('P001', 'CHN001') | turnover 0.02699 | upper limit 0.02533 | units sold 1596 | average stock 59129 | typical pair: units sold 1125 , average stock 68729
2022  all pairs: 0.01638 | without pairs above the limit: 0.01633
2023  all pairs: 0.01294 | without pairs above the limit: 0.01294
2024  all pairs: 0.01034 | without p

In [23]:
s = sd.assign(m=pd.to_datetime(sd['delivery_date']).dt.to_period('M'))
r = pr.assign(m=pd.to_datetime(pr['received_date']).dt.to_period('M'))
mon73 = pd.period_range('2019-01', '2025-01', freq='M')

print('sd delivery_date: first', pd.to_datetime(sd['delivery_date']).min().date(),
      '| last', pd.to_datetime(sd['delivery_date']).max().date())
print('pr received_date: first', pd.to_datetime(pr['received_date']).min().date(),
      '| last', pd.to_datetime(pr['received_date']).max().date())

sm = s.groupby('m')['quantity'].sum().reindex(mon73)
rm = r.groupby('m')['quantity'].sum().reindex(mon73)
print('\nmonths expected:', len(mon73), '| months with deliveries:', int(sm.notna().sum()),
      '| months with receipts:', int(rm.notna().sum()))
print('smallest month 2019-01 to 2024-12: units delivered', int(sm.loc['2019-01':'2024-12'].min()),
      '| units received', int(rm.loc['2019-01':'2024-12'].min()))
print('January 2025 (partial): units delivered', int(sm.loc['2025-01']), '| units received', int(rm.loc['2025-01']))
print('rows outside the 73 months: sd', int((~s['m'].isin(mon73)).sum()), '| pr', int((~r['m'].isin(mon73)).sum()))
print('stock table shape:', stock.shape, '| empty cells:', int(stock.isna().sum().sum()))

g_s = s[s['m'] <= pd.Period('2024-12', 'M')].groupby(['product_id', 'branch_id', 'm']).size()
g_r = r[r['m'] <= pd.Period('2024-12', 'M')].groupby(['product_id', 'branch_id', 'm']).size()
print('\ninformation: pair-months with no delivery (of 180 x 72 = 12960):', 12960 - len(g_s))
print('information: pair-months with no receipt  (of 180 x 72 = 12960):', 12960 - len(g_r))

sd delivery_date: first 2019-01-03 | last 2025-01-13
pr received_date: first 2019-01-11 | last 2025-01-28

months expected: 73 | months with deliveries: 73 | months with receipts: 73
smallest month 2019-01 to 2024-12: units delivered 11734 | units received 143820
January 2025 (partial): units delivered 3372 | units received 194954
rows outside the 73 months: sd 0 | pr 0
stock table shape: (180, 73) | empty cells: 0

information: pair-months with no delivery (of 180 x 72 = 12960): 10
information: pair-months with no receipt  (of 180 x 72 = 12960): 4


In [24]:
datecol = [c for c in L.columns if 'date' in c.lower()][0]
print('ledger date column used:', datecol)

t24 = turn[2024]
med = (t24 - t24.median()).abs().idxmin()
picks = [('P027', 'CHN001', 'fastest in 2024'), ('P005', 'HYD001', 'slowest in 2024'),
         (med[0], med[1], 'closest to the median in 2024')]

mon = pd.period_range('2019-01', '2024-12', freq='M')
for p, b, label in picks:
    print('\n' + p, b, '-', label)
    o = int(inv[(inv['product_id'] == p) & (inv['branch_id'] == b)]['opening_stock'].iloc[0])
    s1 = sd[(sd['product_id'] == p) & (sd['branch_id'] == b)].copy()
    r1 = pr[(pr['product_id'] == p) & (pr['branch_id'] == b)].copy()
    a1 = L[(L['product_id'] == p) & (L['branch_id'] == b) &
           (L['reference_id'].astype(str).str.startswith('ADJ'))].copy()
    s1['m'] = pd.to_datetime(s1['delivery_date']).dt.to_period('M')
    r1['m'] = pd.to_datetime(r1['received_date']).dt.to_period('M')
    a1['m'] = pd.to_datetime(a1[datecol]).dt.to_period('M')
    net = (r1.groupby('m')['quantity'].sum().reindex(mon).fillna(0)
           - s1.groupby('m')['quantity'].sum().reindex(mon).fillna(0)
           + a1.groupby('m')['change'].sum().reindex(mon).fillna(0))
    e24 = (o + net.cumsum()).loc['2024-01':'2024-12']
    sold24 = int(s1[s1['m'].dt.year == 2024]['quantity'].sum())
    print('opening stock', o, '| sales lines', len(s1), '| receipt lines', len(r1), '| adjustment rows', len(a1))
    print('units sold 2024:', sold24, '| units received 2024:', int(r1[r1['m'].dt.year == 2024]['quantity'].sum()))
    print('12 month-end stocks 2024, rebuilt from raw rows:', [int(x) for x in e24.values])
    print('12 month-end stocks 2024, stock table         :', [int(x) for x in st.loc[(p, b)].iloc[60:72]])
    print('month-ends equal:', [int(x) for x in e24.values] == [int(x) for x in st.loc[(p, b)].iloc[60:72]])
    print('turnover 2024, raw rows:', round(sold24 / e24.mean(), 5), '| turnover table:', round(turn.loc[(p, b), 2024], 5))

ledger date column used: movement_date

P027 CHN001 - fastest in 2024
opening stock 84 | sales lines 805 | receipt lines 754 | adjustment rows 8
units sold 2024: 1538 | units received 2024: 17883
12 month-end stocks 2024, rebuilt from raw rows: [93711, 94562, 95920, 96982, 98127, 99063, 101083, 102180, 104330, 105560, 106791, 107621]
12 month-end stocks 2024, stock table         : [93711, 94562, 95920, 96982, 98127, 99063, 101083, 102180, 104330, 105560, 106791, 107621]
month-ends equal: True
turnover 2024, raw rows: 0.0153 | turnover table: 0.0153

P005 HYD001 - slowest in 2024
opening stock 178 | sales lines 543 | receipt lines 805 | adjustment rows 19
units sold 2024: 714 | units received 2024: 20561
12 month-end stocks 2024, rebuilt from raw rows: [103234, 105169, 107042, 108175, 109746, 111288, 113583, 116085, 116991, 119162, 120228, 121716]
12 month-end stocks 2024, stock table         : [103234, 105169, 107042, 108175, 109746, 111288, 113583, 116085, 116991, 119162, 120228, 1217

In [25]:
o = sd.merge(so_h[['so_id', 'order_date']], on='so_id', how='left')
print('sd rows:', len(sd), '| after merge:', len(o), '| lines with no order date:', int(o['order_date'].isna().sum()))
o['oy'] = pd.to_datetime(o['order_date']).dt.year
by_order = o.groupby('oy')['quantity'].sum()
by_deliv = sd.groupby('year')['quantity'].sum()
print('units by order-date year:', {int(k): int(v) for k, v in by_order.items()})

print('\nyear | units by delivery date | units by order date | turnover (delivery) | turnover (order) | difference %')
for y in range(2019, 2025):
    a = avg2[y]
    print(y, '|', int(by_deliv[y]), '|', int(by_order[y]), '|', round(by_deliv[y] / a, 5), '|',
          round(by_order[y] / a, 5), '|', round((by_order[y] - by_deliv[y]) / by_deliv[y] * 100, 2))

sd rows: 117656 | after merge: 117656 | lines with no order date: 0
units by order-date year: {2019: 207925, 2020: 212007, 2021: 208269, 2022: 202242, 2023: 204638, 2024: 199858}

year | units by delivery date | units by order date | turnover (delivery) | turnover (order) | difference %
2019 | 204586 | 207925 | 0.11373 | 0.11558 | 1.63
2020 | 211190 | 212007 | 0.03982 | 0.03998 | 0.39
2021 | 206935 | 208269 | 0.02325 | 0.0234 | 0.64
2022 | 202716 | 202242 | 0.01638 | 0.01634 | -0.23
2023 | 205578 | 204638 | 0.01294 | 0.01288 | -0.46
2024 | 200562 | 199858 | 0.01034 | 0.01031 | -0.35


In [26]:
o['lag'] = (pd.to_datetime(o['delivery_date']) - pd.to_datetime(o['order_date'])).dt.days
print('sd rows:', len(o), '| delivery before order date:', int((o['lag'] < 0).sum()))
print('days from order to delivery: min', int(o['lag'].min()), '| median', int(o['lag'].median()), '| max', int(o['lag'].max()))
cross = o[o['oy'] != o['year']]
print('lines delivered in a different year than ordered:', len(cross), '| units:', int(cross['quantity'].sum()))
print('units ordered in each year but delivered in a later year:')
print(cross.groupby('oy')['quantity'].sum())
print('expected for 2019 (207925 - 204586):', 207925 - 204586)
print('orders placed before 2019-01-01:', int((pd.to_datetime(o['order_date']) < '2019-01-01').sum()))

sd rows: 117656 | delivery before order date: 0
days from order to delivery: min 0 | median 8 | max 15
lines delivered in a different year than ordered: 2482 | units: 25449
units ordered in each year but delivered in a later year:
oy
2019    3339
2020    4156
2021    5490
2022    5016
2023    4076
2024    3372
Name: quantity, dtype: int64
expected for 2019 (207925 - 204586): 3339
orders placed before 2019-01-01: 0


In [27]:
mon73 = pd.period_range('2019-01', '2025-01', freq='M')
idx = pd.MultiIndex.from_frame(inv[['product_id', 'branch_id']])
op = inv.set_index(['product_id', 'branch_id'])['opening_stock']

lg = L[['product_id', 'branch_id', datecol, 'running_balance']].copy()
lg['m'] = pd.to_datetime(lg[datecol]).dt.to_period('M')
last = lg.groupby(['product_id', 'branch_id', 'm'])['running_balance'].last().unstack('m')
last = last.reindex(idx).reindex(columns=mon73).ffill(axis=1)
last = last.apply(lambda c: c.fillna(op))

tot = last.sum()
print('ledger-based stock table:', last.shape, '| empty cells:', int(last.isna().sum().sum()))
print('ledger-based stock, all pairs, 2025-01:', int(tot.loc['2025-01']), '(KPI 8 ledger-based stock: 19305994)')

avgL = tot.groupby(tot.index.year).mean()
print('\nyear | units sold | average stock, ledger-based | average stock, order-line | turnover, ledger-based | turnover, order-line | difference %')
for y in range(2019, 2025):
    print(y, '|', int(by_deliv[y]), '|', int(round(avgL[y])), '|', int(round(avg2[y])), '|',
          round(by_deliv[y] / avgL[y], 5), '|', round(by_deliv[y] / avg2[y], 5), '|',
          round((by_deliv[y] / avgL[y] - by_deliv[y] / avg2[y]) / (by_deliv[y] / avg2[y]) * 100, 2))

ledger-based stock table: (180, 73) | empty cells: 0
ledger-based stock, all pairs, 2025-01: 19305994 (KPI 8 ledger-based stock: 19305994)

year | units sold | average stock, ledger-based | average stock, order-line | turnover, ledger-based | turnover, order-line | difference %
2019 | 204586 | 1638474 | 1798915 | 0.12486 | 0.11373 | 9.79
2020 | 211190 | 4819706 | 5303361 | 0.04382 | 0.03982 | 10.03
2021 | 206935 | 8100272 | 8899554 | 0.02555 | 0.02325 | 9.87
2022 | 202716 | 11266232 | 12378860 | 0.01799 | 0.01638 | 9.88
2023 | 205578 | 14478424 | 15890459 | 0.0142 | 0.01294 | 9.75
2024 | 200562 | 17675030 | 19389511 | 0.01135 | 0.01034 | 9.7


In [28]:
pdate = [c for c in po_h.columns if 'date' in c.lower() and 'received' not in c.lower()][0]
pst = [c for c in po_h.columns if 'status' in c.lower()][0]
print('po_h columns used:', pdate, '|', pst)

sl = so_l.merge(so_h[['so_id', 'order_status', 'order_date']], on='so_id', how='left')
pl = po_l.merge(po_h[['po_id', pst, pdate]], on='po_id', how='left')
print('so_l rows:', len(so_l), '| after merge:', len(sl), '| po_l rows:', len(po_l), '| after merge:', len(pl))
print('sales lines by status:', sl['order_status'].value_counts().to_dict())
print('purchase lines by status:', pl[pst].value_counts().to_dict())
print('units in cancelled sales lines:', int(sl.loc[sl['order_status'] == 'Cancelled', 'quantity'].sum()),
      '| units in cancelled purchase lines:', int(pl.loc[pl[pst] == 'Cancelled', 'quantity'].sum()))

mon = pd.period_range('2019-01', '2024-12', freq='M')
def monthly(df, datec, mask):
    t = df[mask]
    return t.groupby(pd.to_datetime(t[datec]).dt.to_period('M'))['quantity'].sum().reindex(mon).fillna(0)

cs = monthly(sl, 'order_date', sl['order_status'] == 'Cancelled')
cp = monthly(pl, pdate, pl[pst] == 'Cancelled')
print('cancelled units inside the 72 months: sales', int(cs.sum()), '| purchases', int(cp.sum()))

alt = end_stock + cp.cumsum() - cs.cumsum()
avg_alt = alt.groupby(alt.index.year).mean()
cs_y = cs.groupby(cs.index.year).sum()

print('\nyear | turnover, completed orders only (order date) | turnover, cancelled counted as sold and received | difference %')
for y in range(2019, 2025):
    base = by_order[y] / avg2[y]
    withc = (by_order[y] + cs_y[y]) / avg_alt[y]
    print(y, '|', round(base, 5), '|', round(withc, 5), '|', round((withc - base) / base * 100, 2))

po_h columns used: order_date | po_status
so_l rows: 130402 | after merge: 130402 | po_l rows: 155495 | after merge: 155495
sales lines by status: {'Delivered': 117656, 'Cancelled': 12746}
purchase lines by status: {'Received': 140077, 'Cancelled': 15418}
units in cancelled sales lines: 133595 | units in cancelled purchase lines: 2480593
cancelled units inside the 72 months: sales 133595 | purchases 2480593

year | turnover, completed orders only (order date) | turnover, cancelled counted as sold and received | difference %
2019 | 0.11558 | 0.11421 | -1.19
2020 | 0.03998 | 0.03974 | -0.59
2021 | 0.0234 | 0.02329 | -0.47
2022 | 0.01634 | 0.01636 | 0.16
2023 | 0.01288 | 0.01287 | -0.09
2024 | 0.01031 | 0.01043 | 1.23


In [29]:
yrs6 = list(range(2019, 2025))
print("Table: turn (turnover per product-branch pair per year), rows:", len(turn))

print("Pairs with 2024 turnover below 2019:", (turn[2024] < turn[2019]).sum(), "of", len(turn))

falls = turn[yrs6].diff(axis=1).iloc[:, 1:] < 0
print("Pairs falling in every one of the 5 year steps:", falls.all(axis=1).sum(), "of", len(turn))
print("Pairs falling, step by step:")
print(falls.sum())

tp = sold_py[yrs6].groupby(level=0).sum() / avg[yrs6].groupby(level=0).sum()
tf = tp.diff(axis=1).iloc[:, 1:] < 0
print("Products:", len(tp))
print("Products falling in every one of the 5 year steps:", tf.all(axis=1).sum())
print("Products falling, step by step:")
print(tf.sum())

Table: turn (turnover per product-branch pair per year), rows: 180
Pairs with 2024 turnover below 2019: 180 of 180
Pairs falling in every one of the 5 year steps: 142 of 180
Pairs falling, step by step:
year
2020    180
2021    180
2022    169
2023    167
2024    159
dtype: int64
Products: 30
Products falling in every one of the 5 year steps: 30
Products falling, step by step:
year
2020    30
2021    30
2022    30
2023    30
2024    30
dtype: int64


In [30]:
yrs6 = list(range(2019, 2025))
print("Table: turn, rows:", len(turn))

rises = turn[yrs6].diff(axis=1).iloc[:, 1:] > 0
n = rises.sum(axis=1)
risers = n[n > 0]
print("Pairs with at least one rise:", len(risers))
print("Number of rises per pair (rises : pairs):")
print(risers.value_counts().sort_index())

print("Pairs with a rise, by branch:")
print(risers.groupby(level=1).size())
print("Pairs with a rise, by product (top 5):")
print(risers.groupby(level=0).size().sort_values(ascending=False).head(5))

pct = turn[yrs6].pct_change(axis=1).iloc[:, 1:] * 100
print("Size of the rises in % (each rise-event):")
print(pct.where(rises).stack().describe())

Table: turn, rows: 180
Pairs with at least one rise: 38
Number of rises per pair (rises : pairs):
1    31
2     7
Name: count, dtype: int64
Pairs with a rise, by branch:
branch_id
AHM001     5
CHN001     1
DEL001    11
HYD001     8
KOL001     5
PUN001     8
dtype: int64
Pairs with a rise, by product (top 5):
product_id
P024    4
P002    3
P017    3
P022    2
P009    2
dtype: int64
Size of the rises in % (each rise-event):
count    45.000000
mean      9.997474
std       9.384478
min       0.088702
25%       2.815984
50%       6.679710
75%      14.082680
max      36.025943
dtype: float64


In [31]:
sp = sold_py[yrs6]
ap = avg[yrs6]
print("Tables: sold_py rows", len(sp), "| avg rows", len(ap))

sold_ch = sp.pct_change(axis=1).iloc[:, 1:] * 100
stock_ch = ap.pct_change(axis=1).iloc[:, 1:] * 100

ev = pd.DataFrame({
    "sold_change_%": sold_ch.where(rises).stack(),
    "stock_change_%": stock_ch.where(rises).stack()
})
print("Rise-events:", len(ev))
print(ev.describe())
print("Rise-events where units sold went up:", (ev["sold_change_%"] > 0).sum(), "of", len(ev))
print("Rise-events where average stock went down:", (ev["stock_change_%"] < 0).sum(), "of", len(ev))

print("Typical yearly change over all 180 pairs (median, %):")
print("units sold by year:")
print(sold_ch.median())
print("average stock by year:")
print(stock_ch.median())

Tables: sold_py rows 180 | avg rows 180
Rise-events: 45
       sold_change_%  stock_change_%
count      45.000000       45.000000
mean       39.825891       27.244539
std        12.409588        7.010943
min        20.940649       18.484133
25%        28.038380       21.719860
50%        38.275862       24.444639
75%        47.368421       30.412089
max        68.256334       42.586773
Rise-events where units sold went up: 45 of 45
Rise-events where average stock went down: 0 of 45
Typical yearly change over all 180 pairs (median, %):
units sold by year:
year
2020    5.005368
2021   -2.987770
2022   -4.627284
2023    3.578357
2024   -2.533395
dtype: float64
average stock by year:
2020    196.623222
2021     67.114098
2022     38.590499
2023     28.274832
2024     22.058962
dtype: float64


In [32]:
from scipy.stats import hypergeom

N, K, n = 180, 38, 30
cnt = risers.groupby(level=1).size()
print("Pairs:", N, "| pairs with a rise:", K, "| pairs per branch:", n)
print("Expected per branch if even:", round(K * n / N, 2))

for b, c in cnt.items():
    p_hi = hypergeom.sf(c - 1, N, K, n)
    p_lo = hypergeom.cdf(c, N, K, n)
    print(b, "| pairs with a rise:", c,
          "| chance of this many or more:", round(p_hi, 3),
          "| chance of this few or fewer:", round(p_lo, 3))

Pairs: 180 | pairs with a rise: 38 | pairs per branch: 30
Expected per branch if even: 6.33
AHM001 | pairs with a rise: 5 | chance of this many or more: 0.814 | chance of this few or fewer: 0.352
CHN001 | pairs with a rise: 1 | chance of this many or more: 1.0 | chance of this few or fewer: 0.004
DEL001 | pairs with a rise: 11 | chance of this many or more: 0.024 | chance of this few or fewer: 0.992
HYD001 | pairs with a rise: 8 | chance of this many or more: 0.277 | chance of this few or fewer: 0.855
KOL001 | pairs with a rise: 5 | chance of this many or more: 0.814 | chance of this few or fewer: 0.352
PUN001 | pairs with a rise: 8 | chance of this many or more: 0.277 | chance of this few or fewer: 0.855


In [33]:
print("Table: sold_ch and stock_ch (yearly % change per pair), rows:", len(sold_ch))

print("Std of the yearly change in units sold (%), per branch:")
print(sold_ch.stack().groupby(level=1).std().round(2))

print("Number of pair-years with a sales jump above 20%, per branch (150 pair-years each):")
print((sold_ch > 20).groupby(level=1).sum().sum(axis=1))

print("Median yearly change in units sold (%), per branch and year:")
print(sold_ch.groupby(level=1).median().round(2))

print("Median yearly change in average stock (%), per branch and year:")
print(stock_ch.groupby(level=1).median().round(2))

Table: sold_ch and stock_ch (yearly % change per pair), rows: 180
Std of the yearly change in units sold (%), per branch:
branch_id
AHM001    18.88
CHN001    15.65
DEL001    21.41
HYD001    21.76
KOL001    15.43
PUN001    18.29
dtype: float64
Number of pair-years with a sales jump above 20%, per branch (150 pair-years each):
branch_id
AHM001    23
CHN001    12
DEL001    35
HYD001    28
KOL001    15
PUN001    25
dtype: int64
Median yearly change in units sold (%), per branch and year:
year        2020  2021   2022   2023   2024
branch_id                                  
AHM001     13.39 -5.49 -10.85  12.12 -14.21
CHN001    -15.29 -0.62   1.95   7.04  -7.22
DEL001      6.79 -2.07  15.96  -8.62   5.46
HYD001      6.80 -5.39 -10.09   6.13  -1.54
KOL001      4.05  5.67  -9.19   4.19  -0.14
PUN001      8.77 -5.80  -0.48   3.12   6.48
Median yearly change in average stock (%), per branch and year:
             2020   2021   2022   2023   2024
branch_id                                    
AHM

In [34]:
win = [2022, 2023, 2024]
print("Pair-years per branch in 2022-2024:", 30 * 3)

jumps = (sold_ch[win] > 20).groupby(level=1).sum().sum(axis=1)
ev_b = rises[win].groupby(level=1).sum().sum(axis=1)

print("Sales jumps above 20%, 2022-2024, per branch:")
print(jumps)
print("Rise-events (turnover went up), 2022-2024, per branch:")
print(ev_b)
print("Share of those jumps that became a rise (%):")
print((ev_b / jumps * 100).round(1))

Pair-years per branch in 2022-2024: 90
Sales jumps above 20%, 2022-2024, per branch:
branch_id
AHM001    10
CHN001     7
DEL001    22
HYD001    17
KOL001     7
PUN001    16
dtype: int64
Rise-events (turnover went up), 2022-2024, per branch:
branch_id
AHM001     5
CHN001     1
DEL001    13
HYD001    11
KOL001     5
PUN001    10
dtype: int64
Share of those jumps that became a rise (%):
branch_id
AHM001    50.0
CHN001    14.3
DEL001    59.1
HYD001    64.7
KOL001    71.4
PUN001    62.5
dtype: float64


In [35]:
big = sold_ch[win] > 20
js = sold_ch[win].where(big).stack().groupby(level=1).median()
ss = stock_ch[win].where(big).stack().groupby(level=1).median()
n_big = big.groupby(level=1).sum().sum(axis=1)

out = pd.DataFrame({
    "jumps": n_big,
    "median_jump_%": js.round(1),
    "median_stock_growth_same_pair_years_%": ss.round(1)
})
print("Sales jumps above 20% in 2022-2024, per branch:")
print(out)

Sales jumps above 20% in 2022-2024, per branch:
           jumps  median_jump_%  median_stock_growth_same_pair_years_%
branch_id                                                             
AHM001        10           30.5                                   27.9
CHN001         7           24.4                                   32.8
DEL001        22           33.4                                   37.5
HYD001        17           28.7                                   29.3
KOL001         7           29.7                                   24.4
PUN001        16           30.9                                   24.3


In [36]:
pr_m = pr.groupby(pd.to_datetime(pr["received_date"]).dt.to_period("M"))["quantity"].sum().reindex(months, fill_value=0)
sd_m = sd.groupby(pd.to_datetime(sd["delivery_date"]).dt.to_period("M"))["quantity"].sum().reindex(months, fill_value=0)

s2 = inv["opening_stock"].sum() + (pr_m - sd_m).cumsum()
s1 = stock.sum(axis=0) if stock.shape == (180, 73) else stock.sum(axis=1)
gap = pd.Series(s1.values - s2.values, index=months)

print("Months:", len(gap))
print("Gap (stock table minus rebuild without corrections) in the last month:", gap.iloc[-1])
print("Smallest gap:", gap.min(), "| largest gap:", gap.max())
print("Largest gap as % of the rebuilt stock:", round((gap.abs() / s2.values).max() * 100, 3))
print("Gap at the end of each December:")
print(gap[[m for m in months if m.month == 12]])

print("Structure of adj:", type(adj), getattr(adj, "shape", None))
print(adj.head(3))
print("Ledger columns:", ledger.columns.tolist())

Months: 73
Gap (stock table minus rebuild without corrections) in the last month: 62.0
Smallest gap: -955.0 | largest gap: 597.0
Largest gap as % of the rebuilt stock: 0.079
Gap at the end of each December:
2019-12   -668.0
2020-12   -713.0
2021-12   -341.0
2022-12   -293.0
2023-12    329.0
2024-12     62.0
Freq: M, dtype: float64
Structure of adj: <class 'pandas.core.frame.DataFrame'> (2454, 12)
      movement_id product_id branch_id movement_type movement_date  quantity  \
48   MOV-ADJ-1519       P001    AHM001    ADJUSTMENT    2019-04-14        38   
131  MOV-ADJ-1097       P001    AHM001    ADJUSTMENT    2019-08-29        38   
171   MOV-ADJ-524       P001    AHM001    ADJUSTMENT    2019-11-01        33   

    reference_type reference_id  running_balance  opening_stock     prev  \
48             ADJ     ADJ-1519             4207            236   4245.0   
131            ADJ     ADJ-1097            10990            236  11028.0   
171            ADJ      ADJ-524            13979   

In [37]:
adj_p = pd.to_datetime(adj["movement_date"]).dt.to_period("M")
print("Corrections (adj rows):", len(adj), "| net change:", adj["change"].sum())
print("Corrections dated outside the 73 months:", (~adj_p.isin(months)).sum())

adj_m = adj.groupby(adj_p)["change"].sum().reindex(months, fill_value=0)
cum_adj = adj_m.cumsum()
diff = gap - cum_adj

print("Months compared:", len(diff))
print("Months where the gap differs from the cumulative corrections:", (diff != 0).sum(), "of", len(diff))
print("Largest absolute difference:", diff.abs().max())
print("Cumulative corrections at the end of each December:")
print(cum_adj[[m for m in months if m.month == 12]])

Corrections (adj rows): 2454 | net change: 62.0
Corrections dated outside the 73 months: 0
Months compared: 73
Months where the gap differs from the cumulative corrections: 0 of 73
Largest absolute difference: 0.0
Cumulative corrections at the end of each December:
2019-12   -668.0
2020-12   -713.0
2021-12   -341.0
2022-12   -293.0
2023-12    329.0
2024-12     62.0
Freq: M, Name: change, dtype: float64


In [38]:
inv_i = inv.set_index(["product_id", "branch_id"])["opening_stock"]
four = [("P001", "CHN001", 2019), ("P003", "AHM001", 2020), ("P021", "KOL001", 2021), ("P001", "CHN001", 2022)]

for p, b, Y in four:
    rc = pr[pr["year"] <= Y].groupby(["product_id", "branch_id"])["quantity"].sum()
    dc = sd[sd["year"] <= Y].groupby(["product_id", "branch_id"])["quantity"].sum()
    t = pd.DataFrame({"opening": inv_i, "received_to_date": rc, "delivered_to_date": dc}).fillna(0)
    t["stock_no_corrections"] = t["opening"] + t["received_to_date"] - t["delivered_to_date"]
    print(p + "-" + b + ", up to the end of " + str(Y) + " (pairs in table: " + str(len(t)) + "):")
    print(pd.DataFrame({"this_pair": t.loc[(p, b)], "typical_pair_median": t.median()}).round(0))

P001-CHN001, up to the end of 2019 (pairs in table: 180):
                      this_pair  typical_pair_median
opening                     187                178.0
received_to_date          18870              19630.0
delivered_to_date          1689               1084.0
stock_no_corrections      17368              18741.0
P003-AHM001, up to the end of 2020 (pairs in table: 180):
                      this_pair  typical_pair_median
opening                     238                178.0
received_to_date          36575              40724.0
delivered_to_date          2722               2263.0
stock_no_corrections      34091              38502.0
P021-KOL001, up to the end of 2021 (pairs in table: 180):
                      this_pair  typical_pair_median
opening                     280                178.0
received_to_date          54616              62150.0
delivered_to_date          4164               3390.0
stock_no_corrections      50732              58674.0
P001-CHN001, up to the end of 2

In [39]:
for name in ["cs", "cp", "alt", "avg_alt", "avg2", "by_order"]:
    obj = globals().get(name)
    print("---", name, type(obj).__name__, getattr(obj, "shape", None))
    print(obj.head(8) if hasattr(obj, "head") else obj)

--- cs Series (72,)
2019-01    1835
2019-02    1096
2019-03    1783
2019-04    1812
2019-05    1634
2019-06    2209
2019-07    1756
2019-08    1657
Freq: M, Name: quantity, dtype: int64
--- cp Series (72,)
2019-01    45891
2019-02    22012
2019-03    29257
2019-04    40994
2019-05    29498
2019-06    28571
2019-07    26353
2019-08    19977
Freq: M, Name: quantity, dtype: int64
--- alt Series (72,)
2019-01     209111
2019-02     507816
2019-03     871900
2019-04    1216041
2019-05    1528812
2019-06    1847324
2019-07    2159215
2019-08    2478056
Freq: M, Name: quantity, dtype: int64
--- avg_alt Series (6,)
2019    1.993646e+06
2020    5.879784e+06
2021    9.888390e+06
2022    1.375229e+07
2023    1.763742e+07
2024    2.155288e+07
Name: quantity, dtype: float64
--- avg2 Series (6,)
2019    1.798915e+06
2020    5.303361e+06
2021    8.899554e+06
2022    1.237886e+07
2023    1.589046e+07
2024    1.938951e+07
Name: quantity, dtype: float64
--- by_order Series (6,)
oy
2019    207925
2020   

In [40]:
cs_y = cs.groupby(cs.index.year).sum()
cp_y = cp.groupby(cp.index.year).sum()
print("Cancelled sales units total:", cs.sum(), "| cancelled purchase units total:", cp.sum())

t = pd.DataFrame({
    "sold_completed": by_order,
    "sold_cancelled": cs_y,
    "cancelled_purchases": cp_y,
    "stock_avg": avg2,
    "stock_avg_with_cancelled": avg_alt
})
t["sold_push_%"] = t["sold_cancelled"] / t["sold_completed"] * 100
t["stock_push_%"] = (t["stock_avg_with_cancelled"] / t["stock_avg"] - 1) * 100
t["turnover_completed"] = t["sold_completed"] / t["stock_avg"]
t["turnover_with_cancelled"] = (t["sold_completed"] + t["sold_cancelled"]) / t["stock_avg_with_cancelled"]
t["change_%"] = (t["turnover_with_cancelled"] / t["turnover_completed"] - 1) * 100
t["sold_push minus stock_push"] = t["sold_push_%"] - t["stock_push_%"]

print("Years in table:", len(t))
print(t[["sold_completed", "sold_cancelled", "cancelled_purchases"]])
print(t[["sold_push_%", "stock_push_%", "sold_push minus stock_push", "change_%"]].round(2))
print(t[["turnover_completed", "turnover_with_cancelled"]].round(5))

Cancelled sales units total: 133595 | cancelled purchase units total: 2480593
Years in table: 6
      sold_completed  sold_cancelled  cancelled_purchases
2019          207925           19765               375959
2020          212007           21655               449671
2021          208269           22063               403488
2022          202242           22793               391287
2023          204638           22289               424677
2024          199858           25030               435511
      sold_push_%  stock_push_%  sold_push minus stock_push  change_%
2019         9.51         10.82                       -1.32     -1.19
2020        10.21         10.87                       -0.65     -0.59
2021        10.59         11.11                       -0.52     -0.47
2022        11.27         11.09                        0.18      0.16
2023        10.89         10.99                       -0.10     -0.09
2024        12.52         11.16                        1.37      1.23
      tu

In [41]:
m72 = [m for m in months if m.year <= 2024]
d = pd.Series([m.days_in_month for m in m72], index=m72)
dm = sd_m.loc[m72]
rm = pr_m.loc[m72]
print("Months compared (2019-01 to 2024-12):", len(m72))

print("3 smallest months, units delivered:")
print(dm.nsmallest(3))
print("3 smallest months, units received:")
print(rm.nsmallest(3))

print("Typical month (median), units delivered:", dm.median(), "| units received:", rm.median())
print("Smallest month as % of the typical month, delivered:", round(dm.min() / dm.median() * 100, 1),
      "| received:", round(rm.min() / rm.median() * 100, 1))

print("3 smallest months by units per day, delivered:")
print((dm / d).round(0).nsmallest(3))
print("3 smallest months by units per day, received:")
print((rm / d).round(0).nsmallest(3))

Months compared (2019-01 to 2024-12): 72
3 smallest months, units delivered:
2019-01    11734
2019-02    14442
2023-02    14971
Freq: M, Name: quantity, dtype: int64
3 smallest months, units received:
2019-01    143820
2022-02    253937
2023-06    266164
Freq: M, Name: quantity, dtype: int64
Typical month (median), units delivered: 17229.5 | units received: 309938.5
Smallest month as % of the typical month, delivered: 68.1 | received: 46.4
3 smallest months by units per day, delivered:
2019-01    379.0
2020-10    494.0
2023-06    505.0
Freq: M, dtype: float64
3 smallest months by units per day, received:
2019-01    4639.0
2023-06    8872.0
2022-02    9069.0
Freq: M, dtype: float64


In [42]:
pm = pr.merge(po_h[["po_id", "order_date"]].rename(columns={"order_date": "po_order_date"}),
              on="po_id", how="left", suffixes=("", "_x"))
lag = (pd.to_datetime(pm["received_date"]) - pd.to_datetime(pm["po_order_date"])).dt.days

print("Received PO lines:", len(pm), "| lines with no order date:", pm["po_order_date"].isna().sum())
print("First purchase order date:", pd.to_datetime(pm["po_order_date"]).min().date())
print("Days from order to receipt: smallest", lag.min(), "| median", lag.median(), "| largest", lag.max())
print("Lines received before they were ordered:", (lag < 0).sum())

Received PO lines: 140077 | lines with no order date: 0
First purchase order date: 2019-01-01
Days from order to receipt: smallest 7 | median 18.0 | largest 35
Lines received before they were ordered: 0


In [43]:
su = sold_py.groupby(level=0).sum()
sk = avg.groupby(level=0).sum()
cw2 = pr.groupby("product_id")["line_total"].sum() / pr.groupby("product_id")["quantity"].sum()
ct2 = products.set_index("product_id")["unit_cost"]
print("Products:", len(su), "| received cost rows:", len(cw2), "| products-table cost rows:", len(ct2))

rows = []
for Y in [2019, 2024]:
    s, a = su[Y], sk[Y]
    pt = s / a
    rows.append({
        "year": Y,
        "lowest product turnover": pt.min(),
        "units, no cost": s.sum() / a.sum(),
        "received cost": (s * cw2).sum() / (a * cw2).sum(),
        "products-table cost": (s * ct2).sum() / (a * ct2).sum(),
        "equal weights per product": pt.mean(),
        "highest product turnover": pt.max(),
        "highest / lowest": pt.max() / pt.min()
    })
print(pd.DataFrame(rows).set_index("year").round(5).T)

Products: 30 | received cost rows: 30 | products-table cost rows: 30
year                          2019     2024
lowest product turnover    0.09492  0.00932
units, no cost             0.11376  0.01034
received cost              0.11418  0.01023
products-table cost        0.11419  0.01022
equal weights per product  0.11398  0.01035
highest product turnover   0.12962  0.01156
highest / lowest           1.36553  1.24043
